# Ingest Data
- customer
- customer_type
- product
- product_category
- order

In [0]:
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("db_name", "")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql import Row


catalog_name = dbutils.widgets.get("catalog_name")
db_name = dbutils.widgets.get("db_name")

spark.sql(f"USE CATALOG {catalog_name}")

spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")
spark.sql(f"USE {db_name}")

# Cleanup old objects if they already exist
spark.sql(f"DROP VIEW IF EXISTS {db_name}.vw_top_5_products_by_review")
spark.sql(f"DROP VIEW IF EXISTS {db_name}.vw_top_5_products_by_lowest_revenue")
spark.sql(f"DROP TABLE IF EXISTS {db_name}.top_5_customers_by_revenue")
spark.sql(f"DROP TABLE IF EXISTS {db_name}.order_tbl")
spark.sql(f"DROP TABLE IF EXISTS {db_name}.product")
spark.sql(f"DROP TABLE IF EXISTS {db_name}.customer")

In [0]:
customer_type_data = [
    (1, " regular ", "2026-07-01 09:00:00"),
    (2, "PREMIUM", "2026-07-01 09:00:00"),
    (2, " Premium ", "2026-07-02 09:00:00"),
    (3, "corporate", "2026-07-01 09:00:00")
]

customer_type_schema = StructType([
    StructField("customer_type_id", IntegerType(), True),
    StructField("customer_type_name", StringType(), True),
    StructField("updated_ts", StringType(), True)
])

customer_type_raw_df = spark.createDataFrame(customer_type_data, customer_type_schema)

customer_data = [
    (101, " alice ", "ALICE@MAIL.COM ", " bangalore ", 1, "2026-07-01 10:00:00"),
    (101, "Alice", "alice@mail.com", "Bangalore", 1, "2026-07-03 09:00:00"),
    (102, "Bob", "bob@mail.com", "Chennai", 2, "2026-07-02 11:00:00"),
    (103, "Charlie", "charlie@mail.com", "mumbai", 3, "2026-07-02 12:00:00"),
    (104, "Diana", "diana@mail.com", "Pune", 2, "2026-07-02 12:30:00"),
    (105, "Evan", None, "Hyderabad", 2, "2026-07-02 13:00:00"),
    (106, "Frank", "frank@mail.com", "Delhi", 99, "2026-07-02 13:30:00"),
    (107, "Grace", " grace@mail.com ", " Bangalore ", 1, "2026-07-02 14:00:00"),
    (108, "Henry", "henry@mail.com", "Chennai", 2, "2026-07-02 14:30:00")
]

customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("customer_type_id", IntegerType(), True),
    StructField("updated_ts", StringType(), True)
])

customer_raw_df = spark.createDataFrame(customer_data, customer_schema)

display(customer_type_raw_df)
display(customer_raw_df)

In [0]:
product_category_data = [
    (10, " electronics ", "2026-07-01 09:00:00"),
    (20, "HOME", "2026-07-01 09:00:00"),
    (20, " Home ", "2026-07-02 09:00:00"),
    (30, "grocery", "2026-07-01 09:00:00")
]

product_category_schema = StructType([
    StructField("product_category_id", IntegerType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("updated_ts", StringType(), True)
])

product_category_raw_df = spark.createDataFrame(product_category_data, product_category_schema)

product_data = [
    (1001, " laptop ", 10, 70000.0, "2026-07-01 10:00:00"),
    (1001, "Laptop", 10, 72000.0, "2026-07-03 09:00:00"),
    (1002, "Mobile", 10, 30000.0, "2026-07-02 10:00:00"),
    (1003, "Mixer", 20, 4500.0, "2026-07-02 10:30:00"),
    (1004, "Chair", 20, 2500.0, "2026-07-02 11:00:00"),
    (1005, "Rice Bag", 30, 1200.0, "2026-07-02 11:30:00"),
    (1006, "Headphones", 10, 0.0, "2026-07-02 12:00:00"),
    (1007, "Desk", 20, 6500.0, "2026-07-02 12:30:00")
]

product_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("product_category_id", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("updated_ts", StringType(), True)
])

product_raw_df = spark.createDataFrame(product_data, product_schema)

display(product_category_raw_df)
display(product_raw_df)

In [0]:
order_data = [
    (5001, 101, 1001, 1, 72000.0, 10, "2026-07-03 10:00:00", "2026-07-03 10:00:00"),
    (5001, 101, 1001, 1, 72000.0, 12, "2026-07-03 10:00:00", "2026-07-03 10:05:00"),
    (5002, 102, 1002, 2, 30000.0, 8, "2026-07-04 11:00:00", "2026-07-04 11:00:00"),
    (5003, 103, 1003, 1, 4500.0, 5, "2026-07-04 12:00:00", "2026-07-04 12:00:00"),
    (5004, 104, 1004, 4, 2500.0, 20, "2026-07-05 13:00:00", "2026-07-05 13:00:00"),
    (5005, 102, 1005, 5, 1200.0, 15, "2026-07-05 14:00:00", "2026-07-05 14:00:00"),
    (5006, 104, 1007, 1, 6500.0, 2, "2026-07-05 15:00:00", "2026-07-05 15:00:00"),
    (5007, 105, 1002, 1, 30000.0, 6, "2026-07-06 10:00:00", "2026-07-06 10:00:00"),
    (5008, 101, 1006, 1, 0.0, 4, "2026-07-06 11:00:00", "2026-07-06 11:00:00"),
    (5009, 103, 1005, 0, 1200.0, 1, "2026-07-06 12:00:00", "2026-07-06 12:00:00"),
    (5010, 104, 1001, 1, 72000.0, 30, "2026-07-06 13:00:00", "2026-07-06 13:00:00"),
    (5011, 102, 1007, 3, 6500.0, 1, "2026-07-06 14:00:00", "2026-07-06 14:00:00"),
    (5012, 103, 1002, 1, 30000.0, 9, "2026-07-06 15:00:00", "2026-07-06 15:00:00"),
    (5013, 107, 1005, 20, 1200.0, 3, "2026-07-06 16:00:00", "2026-07-06 16:00:00"),
    (5014, 108, 1004, 2, 2500.0, 7, "2026-07-06 17:00:00", "2026-07-06 17:00:00")
]

order_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("review_count", IntegerType(), True),
    StructField("order_ts", StringType(), True),
    StructField("updated_ts", StringType(), True)
])

order_raw_df = spark.createDataFrame(order_data, order_schema)

display(order_raw_df)

In [0]:
# Clean customer type
customer_type_clean_df = (
    customer_type_raw_df
    .withColumn("customer_type_name", F.initcap(F.trim(F.col("customer_type_name"))))
    .withColumn("updated_ts", F.to_timestamp("updated_ts"))
    .dropna(subset=["customer_type_id", "customer_type_name"])
)

w_customer_type = Window.partitionBy("customer_type_id").orderBy(F.col("updated_ts").desc())

customer_type_dedup_df = (
    customer_type_clean_df
    .withColumn("rn", F.row_number().over(w_customer_type))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Clean customer
customer_clean_df = (
    customer_raw_df
    .withColumn("customer_name", F.initcap(F.trim(F.col("customer_name"))))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("updated_ts", F.to_timestamp("updated_ts"))
    .dropna(subset=["customer_id", "customer_name", "email", "city", "customer_type_id"])
)

w_customer = Window.partitionBy("customer_id").orderBy(F.col("updated_ts").desc())

customer_dedup_df = (
    customer_clean_df
    .withColumn("rn", F.row_number().over(w_customer))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Keep only valid customer_type_id values
customer_final_df = (
    customer_dedup_df.alias("c")
    .join(customer_type_dedup_df.alias("ct"), on="customer_type_id", how="inner")
    .select(
        "c.customer_id",
        "c.customer_name",
        "c.email",
        "c.city",
        "c.customer_type_id",
        F.col("ct.customer_type_name").alias("customer_type_name"),
        "c.updated_ts"
    )
)

customer_final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{db_name}.customer")

display(customer_final_df)

In [0]:
# Clean product category
product_category_clean_df = (
    product_category_raw_df
    .withColumn("product_category_name", F.initcap(F.trim(F.col("product_category_name"))))
    .withColumn("updated_ts", F.to_timestamp("updated_ts"))
    .dropna(subset=["product_category_id", "product_category_name"])
)

w_product_category = Window.partitionBy("product_category_id").orderBy(F.col("updated_ts").desc())

product_category_dedup_df = (
    product_category_clean_df
    .withColumn("rn", F.row_number().over(w_product_category))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Clean product
product_clean_df = (
    product_raw_df
    .withColumn("product_name", F.initcap(F.trim(F.col("product_name"))))
    .withColumn("updated_ts", F.to_timestamp("updated_ts"))
    .filter(F.col("unit_price") > 0)
    .dropna(subset=["product_id", "product_name", "product_category_id", "unit_price"])
)

w_product = Window.partitionBy("product_id").orderBy(F.col("updated_ts").desc())

product_dedup_df = (
    product_clean_df
    .withColumn("rn", F.row_number().over(w_product))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Keep only valid category values
product_final_df = (
    product_dedup_df.alias("p")
    .join(product_category_dedup_df.alias("pc"), on="product_category_id", how="inner")
    .select(
        "p.product_id",
        "p.product_name",
        "p.product_category_id",
        F.col("pc.product_category_name").alias("product_category_name"),
        "p.unit_price",
        "p.updated_ts"
    )
)

product_final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{db_name}.product")

display(product_final_df)

In [0]:
customer_df = spark.table(f"{db_name}.customer")
product_df = spark.table(f"{db_name}.product")

order_clean_df = (
    order_raw_df
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("updated_ts", F.to_timestamp("updated_ts"))
    .filter(
        (F.col("quantity") > 0) &
        (F.col("unit_price") > 0) &
        (F.col("review_count") >= 0)
    )
    .dropna(subset=["order_id", "customer_id", "product_id", "order_ts"])
)

w_order = Window.partitionBy("order_id").orderBy(F.col("updated_ts").desc())

order_dedup_df = (
    order_clean_df
    .withColumn("rn", F.row_number().over(w_order))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Keep only valid customer_id and product_id
order_final_df = (
    order_dedup_df.alias("o")
    .join(customer_df.select("customer_id").alias("c"), on="customer_id", how="inner")
    .join(product_df.select("product_id").alias("p"), on="product_id", how="inner")
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("revenue", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .select(
        "o.order_id",
        "o.customer_id",
        "o.product_id",
        "o.quantity",
        "o.unit_price",
        "o.review_count",
        "o.order_ts",
        "order_date",
        "revenue",
        "o.updated_ts"
    )
)

order_final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{db_name}.order_tbl")

display(order_final_df)

In [0]:
display(spark.table(f"{db_name}.customer"))
display(spark.table(f"{db_name}.product"))
display(spark.table(f"{db_name}.order_tbl"))